# PlantVillage classification with folder-based generators
This notebook expects PlantVillage image folders and keeps the supplied train/validation split. It does not fabricate a test set: if no labelled test folder is present, only validation metrics are reported. Set `PLANTVILLAGE_ROOT` to the dataset root before running. The optional EfficientNet path is skipped unless `PLANT_DISEASE_RUN_EFFICIENTNET=1`.


In [ ]:
# ruff: noqa: E402
import hashlib
import os
import random
from pathlib import Path

SEED = int(os.environ.get("PLANT_DISEASE_SEED", "42"))
os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")
random.seed(SEED)

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import tensorflow as tf
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import BatchNormalization, Conv2D, Dense, Dropout, Flatten
from tensorflow.keras.optimizers import Adamax
from tensorflow.keras.preprocessing.image import ImageDataGenerator

np.random.seed(SEED)
tf.random.set_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except (AttributeError, RuntimeError):
    pass


def env_int(name: str, default: int, minimum: int = 1) -> int:
    """Read a positive integer environment setting with an actionable error."""
    raw = os.environ.get(name, str(default))
    try:
        value = int(raw)
    except ValueError as exc:
        raise ValueError(f"{name} must be an integer; received {raw!r}") from exc
    if value < minimum:
        raise ValueError(f"{name} must be >= {minimum}; received {value}")
    return value


In [ ]:
DATA_ROOT = Path(os.environ.get("PLANTVILLAGE_ROOT", "data/PlantVillage")).expanduser()
OUTPUT_DIR = Path(os.environ.get("PLANT_DISEASE_OUTPUT_DIR", "outputs")).expanduser()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def resolve_split(root: Path, split: str, required: bool = True):
    candidates = [
        root / "New Plant Diseases Dataset(Augmented)" / "New Plant Diseases Dataset(Augmented)" / split,
        root / split,
    ]
    for candidate in candidates:
        if candidate.is_dir() and any(child.is_dir() for child in candidate.iterdir()):
            return candidate
    if required:
        checked = ", ".join(str(path) for path in candidates)
        raise FileNotFoundError(
            f"Could not find a labelled '{split}' split. Set PLANTVILLAGE_ROOT to the "
            f"extracted dataset root; checked: {checked}"
        )
    return None


def make_frame(split_dir: Path) -> pd.DataFrame:
    rows = []
    unreadable = []
    for class_dir in sorted(path for path in split_dir.iterdir() if path.is_dir()):
        for image_path in sorted(class_dir.iterdir()):
            if not (image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS):
                continue
            try:
                with Image.open(image_path) as image:
                    image.verify()
            except (OSError, SyntaxError):
                unreadable.append(str(image_path))
                continue
            rows.append({"filename": str(image_path), "label": class_dir.name})
    if unreadable:
        sample = ", ".join(unreadable[:3])
        raise ValueError(f"Unreadable image(s) found below {split_dir}; examples: {sample}")
    frame = pd.DataFrame(rows, columns=["filename", "label"])
    if frame.empty:
        raise ValueError(f"No supported images found below {split_dir}")
    return frame


def content_digests(frame: pd.DataFrame) -> dict[str, str]:
    """Hash labelled files so byte-identical copies cannot cross the split."""
    return {
        hashlib.sha256(Path(filename).read_bytes()).hexdigest(): filename
        for filename in frame["filename"]
    }


train_dir = resolve_split(DATA_ROOT, "train")
valid_dir = resolve_split(DATA_ROOT, "valid", required=False) or resolve_split(DATA_ROOT, "validation")
df_train, df_valid = make_frame(train_dir), make_frame(valid_dir)
train_classes, valid_classes = set(df_train.label), set(df_valid.label)
if len(train_classes) < 2:
    raise ValueError("At least two classes are required for classification")
if train_classes != valid_classes:
    raise ValueError(
        "Train/validation class mismatch; "
        f"missing in validation: {sorted(train_classes - valid_classes)}; "
        f"extra: {sorted(valid_classes - train_classes)}"
    )
train_digests, valid_digests = content_digests(df_train), content_digests(df_valid)
overlap = set(train_digests).intersection(valid_digests)
if overlap:
    digest = sorted(overlap)[0]
    raise ValueError(
        "Duplicate image content detected across train/validation: "
        f"{train_digests[digest]} and {valid_digests[digest]}"
    )
print(f"Train: {df_train.shape}; validation: {df_valid.shape}; classes: {len(train_classes)}")


In [ ]:
image_size = (224, 224)
batch_size = env_int("PLANT_DISEASE_BATCH_SIZE", 32)
train_generator_config = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
)
validation_generator_config = ImageDataGenerator(rescale=1.0 / 255)
train_gen = train_generator_config.flow_from_dataframe(
    df_train,
    x_col="filename",
    y_col="label",
    target_size=image_size,
    class_mode="categorical",
    batch_size=batch_size,
    shuffle=True,
    seed=SEED,
    classes=sorted(train_classes),
)
valid_gen = validation_generator_config.flow_from_dataframe(
    df_valid,
    x_col="filename",
    y_col="label",
    target_size=image_size,
    class_mode="categorical",
    batch_size=batch_size,
    shuffle=False,
    classes=sorted(train_classes),
)
class_dict = train_gen.class_indices
labels = [name for name, _ in sorted(class_dict.items(), key=lambda item: item[1])]
if valid_gen.class_indices != class_dict:
    raise ValueError("Train/validation generator class-index mappings differ")


In [ ]:
model = Sequential([
    Conv2D(64, (3, 3), padding="same", activation="relu", input_shape=(*image_size, 3)),
    Conv2D(64, (3, 3), padding="same", activation="relu"), BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), padding="same", activation="relu"),
    Conv2D(128, (3, 3), padding="same", activation="relu"), BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    Conv2D(256, (3, 3), padding="same", activation="relu"),
    Conv2D(256, (3, 3), padding="same", activation="relu"), BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    Flatten(), Dense(256, activation="relu"), Dense(64, activation="relu"),
    Dropout(0.3), Dense(len(labels), activation="softmax"),
], name="folder_cnn")
model.compile(optimizer=Adamax(learning_rate=0.001), loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()


In [ ]:
epochs = env_int("PLANT_DISEASE_EPOCHS", 40)
history = model.fit(
    train_gen,
    epochs=epochs,
    validation_data=valid_gen,
    callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
    verbose=1,
)
validation_metrics = model.evaluate(valid_gen, verbose=0, return_dict=True)
print("Validation metrics:", validation_metrics)
model.save(OUTPUT_DIR / "folder_cnn.keras")


In [ ]:
valid_gen.reset()
predictions = model.predict(valid_gen, verbose=0)
y_pred = np.argmax(predictions, axis=1)
y_true = valid_gen.classes
if len(y_pred) != len(y_true):
    raise RuntimeError(f"Prediction count {len(y_pred)} does not match labels {len(y_true)}")
cm = confusion_matrix(y_true, y_pred, labels=range(len(labels)))
plt.figure(figsize=(max(10, len(labels) * 0.45), max(8, len(labels) * 0.35)))
sns.heatmap(cm, annot=False, cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Validation confusion matrix")
plt.tight_layout()
print(classification_report(y_true, y_pred, labels=range(len(labels)), target_names=labels, zero_division=0))


In [ ]:
def predict_image(img_path: str | Path, model_to_use=model):
    """Predict one local image; raises a clear error instead of hiding missing files."""
    path = Path(img_path).expanduser()
    if not path.is_file():
        raise FileNotFoundError(f"Prediction image does not exist: {path}")
    image = Image.open(path).convert("RGB").resize(image_size)
    batch = np.asarray(image, dtype=np.float32)[None, ...] / 255.0
    probabilities = model_to_use.predict(batch, verbose=0)[0]
    result = pd.Series(probabilities, index=labels).sort_values(ascending=False)
    return result

# Example (opt-in): set PLANTVILLAGE_EXAMPLE_IMAGE to a real local image path.
example = os.environ.get("PLANTVILLAGE_EXAMPLE_IMAGE")
if example:
    print(predict_image(example))


In [ ]:
# Optional transfer-learning model. It is skipped unless explicitly enabled.
RUN_EFFICIENTNET = os.environ.get("PLANT_DISEASE_RUN_EFFICIENTNET", "0") == "1"
if RUN_EFFICIENTNET:
    # By default weights=None avoids an implicit network download. ImageNet weights
    # freeze the backbone for feature extraction; random weights keep it trainable.
    use_imagenet = os.environ.get("PLANT_DISEASE_IMAGENET_WEIGHTS", "0") == "1"
    base_model = tf.keras.applications.EfficientNetB3(
        include_top=False,
        weights="imagenet" if use_imagenet else None,
        input_shape=(*image_size, 3),
        pooling="max",
    )
    base_model.trainable = not use_imagenet
    model_pretrained = Sequential(
        [
            base_model,
            Dense(256, activation="relu"),
            Dropout(0.3),
            Dense(len(labels), activation="softmax"),
        ],
        name="efficientnet_classifier",
    )
    model_pretrained.compile(
        optimizer=Adamax(learning_rate=0.001),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
else:
    print("Skipping EfficientNet; set PLANT_DISEASE_RUN_EFFICIENTNET=1 to enable it.")


In [ ]:
if RUN_EFFICIENTNET:
    # This second training run uses the same fixed validation split.
    pretrained_epochs = env_int("PLANT_DISEASE_PRETRAINED_EPOCHS", 5)
    history_final = model_pretrained.fit(
        train_gen,
        epochs=pretrained_epochs,
        validation_data=valid_gen,
        verbose=1,
    )
    print(model_pretrained.evaluate(valid_gen, verbose=0, return_dict=True))
    model_pretrained.save(OUTPUT_DIR / "efficientnet_classifier.keras")
